# IE212 — Big Data: PySpark ETL, MLlib ALS, Distributed Runner & Benchmark

Notebook chạy trọn gói phần `bigdata/` trên **Google Colab** (hoặc **Kaggle Notebook**) mà không cần cài đặt gì lên máy cá nhân.

**Yêu cầu trước khi chạy:** Đã upload 2 file `reviews_anonymized.jsonl` và `products.jsonl` lên Google Drive nhóm (vd `MyDrive/tiki-recsys/data/raw/`) hoặc Kaggle Dataset.

## 1. Khởi tạo môi trường & cài đặt thư viện

In [ ]:
import os, sys, platform
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = Path("/kaggle/working").exists()
print(f"Colab={IS_COLAB} | Kaggle={IS_KAGGLE} | CPU={os.cpu_count()} | Python={platform.python_version()}")
!java -version

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# Cập nhật REPO_URL và BRANCH nếu cần
REPO_URL = "https://github.com/Tran-Le-Thien-Bao/Tiki---RecSys---Data.git"
BRANCH = "main"
REPO_DIR = Path("/content/Tiki-RecSys-Data" if IS_COLAB else "/kaggle/working/Tiki-RecSys-Data")

if not REPO_DIR.exists():
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
!pip install -q -r requirements.txt -r requirements-recsys.txt -r requirements-bigdata.txt

## 2. Cấu hình đường dẫn dữ liệu & kết quả

- `DATA_DIR`: thư mục chứa `raw/reviews_anonymized.jsonl` và `raw/products.jsonl`.
- `RESULTS_DIR`: thư mục lưu JSON & biểu đồ trên Drive để không bị mất khi hết phiên Colab.
- `WORK_DIR`: thư mục SSD cục bộ của VM (`/content/tmp_bench` hoặc `/kaggle/working/tmp_bench`) cho benchmark để tránh nghẽn I/O Drive.

In [ ]:
if IS_COLAB:
    DATA_DIR = Path("/content/drive/MyDrive/tiki-recsys/data")
    RESULTS_DIR = Path("/content/drive/MyDrive/tiki-recsys/results/bigdata")
    WORK_DIR = Path("/content/tmp_bench")
elif IS_KAGGLE:
    DATA_DIR = Path("/kaggle/input/tiki-recsys-raw")
    RESULTS_DIR = Path("/kaggle/working/results/bigdata")
    WORK_DIR = Path("/kaggle/working/tmp_bench")
else:
    DATA_DIR = Path("./data")
    RESULTS_DIR = Path("./results/bigdata")
    WORK_DIR = Path("./data/bigdata/tmp_bench")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("DATA_DIR   =", DATA_DIR)
print("RESULTS_DIR=", RESULTS_DIR)
print("WORK_DIR   =", WORK_DIR)
assert (DATA_DIR / "raw/reviews_anonymized.jsonl").exists(), "Thiếu data/raw/reviews_anonymized.jsonl"
assert (DATA_DIR / "raw/products.jsonl").exists(), "Thiếu data/raw/products.jsonl"

## 3. PySpark ETL (Bronze → Silver → Gold) + 3 cờ Window Functions + Parity Check

In [ ]:
!python -m bigdata.etl --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --star all pos45 --k 0 2 3 5

In [ ]:
import json
parity = json.loads((RESULTS_DIR / "parity_check.json").read_text(encoding="utf-8"))
etl_sum = json.loads((RESULTS_DIR / "etl_summary.json").read_text(encoding="utf-8"))
print("Thời gian ETL PySpark:", etl_sum["elapsed_s"], "giây")
print("Parity Check:", json.dumps(parity, ensure_ascii=False, indent=2))
print("Tổng hợp cờ Gold:", json.dumps(etl_sum["flags"], ensure_ascii=False, indent=2))

## 4. Spark MLlib ALS (implicitPrefs=True) & Đối chiếu với implicit ALS

In [ ]:
!python -m bigdata.spark_als --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --k 0 2 --seeds 0 1

In [ ]:
als_sum = json.loads((RESULTS_DIR / "spark_als_summary.json").read_text(encoding="utf-8"))
print(json.dumps(als_sum, ensure_ascii=False, indent=2))

## 5. Distributed RunConfig Runner (chạy song song + resume)

In [ ]:
!python -m bigdata.runner --suite pilot --mode spark --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}"

## 6. Benchmark: Pandas vs PySpark (1×, 5×, 10×), Spark Tuning & Core Scaling

In [ ]:
!python -m bigdata.benchmark --data-dir "{DATA_DIR}" --results-dir "{RESULTS_DIR}" --work-dir "{WORK_DIR}" --scales 1 5 10

In [ ]:
from IPython.display import Image, display

bench = json.loads((RESULTS_DIR / "benchmark_summary.json").read_text(encoding="utf-8"))
print(json.dumps(bench, ensure_ascii=False, indent=2))

for fname in ("bd1_pandas_vs_spark.png", "bd2_spark_tuning.png", "bd3_core_scaling.png"):
    fig_path = RESULTS_DIR / "figures" / fname
    if fig_path.exists():
        display(Image(filename=str(fig_path)))

## 7. Lưu ý trước khi commit kết quả vào repo

- Chỉ đưa các file tổng hợp (`etl_summary.json`, `parity_check.json`, `spark_als_summary.json`, `runner_summary.json`, `benchmark_summary.json`) và 3 hình `results/bigdata/figures/*.png` về repo.
- **Tuyệt đối không** đưa `data/`, `per_user.parquet` hoặc output chứa dữ liệu thô lên Git.
- Nếu sửa notebook này trước khi commit: chọn **Edit → Clear all outputs**.